<a href="https://colab.research.google.com/github/Al-Amin95/Prompt-Injection-Detection-System/blob/main/notebooks/01_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

dataset = load_dataset("deepset/prompt-injections")
print(dataset)

In [ ]:
import pandas as pd

train_df = dataset["train"].to_pandas()
test_df  = dataset["test"].to_pandas()

print("Train shape:", train_df.shape, " Test shape:", test_df.shape)
print("\nLabel counts (train):")
print(train_df["label"].value_counts())
print("\nFirst 10 rows:")
train_df.head(100)

In [ ]:
# Label balance in the test set too
print("Label counts (test):")
print(test_df["label"].value_counts())

# Missing / empty values
print("\nMissing values (train):")
print(train_df.isnull().sum())
print("Empty text rows (train):", (train_df["text"].str.strip() == "").sum())

# Duplicates
print("\nDuplicate full rows (train):", train_df.duplicated().sum())
print("Duplicate text values (train):", train_df["text"].duplicated().sum())

In [ ]:
# Same quality checks on the test set
print("Missing values (test):\n", test_df.isnull().sum())
print("Empty text rows (test):", (test_df["text"].str.strip() == "").sum())
print("Duplicate texts (test):", test_df["text"].duplicated().sum())

# Data-leakage check: any prompt appearing in BOTH train and test?
overlap = set(train_df["text"]) & set(test_df["text"])
print("\nOverlapping prompts between train and test:", len(overlap))

In [ ]:
#Text length analysis
train_df["char_len"] = train_df["text"].str.len()
train_df["word_len"] = train_df["text"].str.split().str.len()

print("Word length stats (train):")
print(train_df["word_len"].describe())
print("\nAverage word length by label (0=SAFE, 1=INJECTION):")
print(train_df.groupby("label")["word_len"].mean())
print("\nLongest prompt (words):", train_df["word_len"].max())

In [ ]:
# Label-balance chart
import matplotlib.pyplot as plt

counts = train_df["label"].value_counts().sort_index()
plt.figure(figsize=(5,4))
plt.bar(["SAFE (0)", "INJECTION (1)"], counts.values, color=["#4caf50", "#e53935"])
plt.title("Train set: SAFE vs INJECTION")
plt.ylabel("Number of prompts")
for i, v in enumerate(counts.values):
    plt.text(i, v + 3, str(v), ha="center")
plt.tight_layout()
plt.show()

In [ ]:
!pip install langdetect -q
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42

def safe_detect(t):
    try: return detect(t)
    except: return "unknown"

train_df["lang"] = train_df["text"].apply(safe_detect)
print("Language distribution (train):")
print(train_df["lang"].value_counts())

In [ ]:
# Common injection keywords
keywords = ["ignore", "forget", "previous", "instructions", "instead",
            "disregard", "system", "prompt", "now", "task"]
print("Keyword frequency — injection vs safe:")
for kw in keywords:
    inj  = train_df[train_df["label"]==1]["text"].str.contains(kw, case=False).sum()
    safe = train_df[train_df["label"]==0]["text"].str.contains(kw, case=False).sum()
    print(f"{kw:14s} injection={inj:3d}   safe={safe:3d}")

In [ ]:
#Length boxplot
import matplotlib.pyplot as plt

safe_len = train_df[train_df["label"]==0]["word_len"]
inj_len  = train_df[train_df["label"]==1]["word_len"]

plt.figure(figsize=(6,4))
plt.boxplot([safe_len, inj_len], labels=["SAFE", "INJECTION"], showfliers=False)
plt.ylabel("Words per prompt")
plt.title("Prompt length: SAFE vs INJECTION")
plt.tight_layout()
plt.show()

In [ ]:
#Example prompts table
import pandas as pd
pd.set_option("display.max_colwidth", None)

examples = pd.DataFrame({
    "label":  ["SAFE"]*3 + ["INJECTION"]*3,
    "prompt": train_df[train_df["label"]==0]["text"].head(3).tolist()
            + train_df[train_df["label"]==1]["text"].head(3).tolist()
})
examples

In [ ]:
#Word cloud
!pip install wordcloud -q
from wordcloud import WordCloud
import matplotlib.pyplot as plt

inj_text = " ".join(train_df[train_df["label"]==1]["text"])
wc = WordCloud(width=800, height=400, background_color="white").generate(inj_text)
plt.figure(figsize=(10,5))
plt.imshow(wc, interpolation="bilinear"); plt.axis("off")
plt.title("Most frequent words in INJECTION prompts")
plt.tight_layout(); plt.show()

In [ ]:
#Common two-word phrases in injections
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(ngram_range=(2,2))
X = vec.fit_transform(train_df[train_df["label"]==1]["text"])
freqs = X.sum(axis=0).A1
top = sorted(zip(vec.get_feature_names_out(), freqs), key=lambda x: -x[1])[:15]
print("Top 15 two-word phrases in INJECTION prompts:")
for phrase, c in top:
    print(f"{phrase:25s} {c}")

In [ ]:
#Label split by language
sub = train_df[train_df["lang"].isin(["en","de"])]
ct = sub.groupby(["lang","label"]).size().unstack(fill_value=0)
ct.columns = ["SAFE (0)", "INJECTION (1)"]
print("Label counts by language (train):")
print(ct)